# 🚀 Workshop Day 1: Data Engineering — Turning Junk into Fuel
## Agentic RAG: From Zero to Hero

**English:** 3 English | **English:** English

---

### 🎯 English

English **Data Engineering Pipeline** English RAG English:

```
📄 English → 🔍 English Duplicate → ✂️ Chunking → 📝 Markdown → 🔢 Embedding → 🗄️ VectorDB → 🔎 Retrieval
```

> **RAG (Retrieval-Augmented Generation)** English LLM English
> English "English" English "English" English


### 🗺️ English Pipeline English

| Step | Section | Task |
|:----:|---------|------|
| 📁 | Input | Raw Documents (PDF, TXT, DOCX) |
| ⬇️ | 1.1 Deduplication | English MD5 Hash |
| ⬇️ | 1.2 Chunking | English |
| ⬇️ | 1.3-1.4 Markdown | English PDF → Markdown (Gemini / Docling) |
| ⬇️ | 1.5 Embedding | English → Vector (multilingual-e5-large) |
| ⬇️ | 1.6 Hybrid Search | English Dense + Sparse (BM25) |
| ⬇️ | 1.7 VectorDB | English Qdrant |
| ⬇️ | 1.8 Indexing | Upsert vectors English Qdrant |
| 🔎 | 1.9 Retrieval | English vectors English query |

> 💡 English **English** English — English RAG English!

---
## 📦 Section 0: English Dependencies

English library English workshop English

In [16]:
%%time
# English libraries English
!pip install -q google-genai \
    'docling>=2.31' \
    'docling-ibm-models>=3.4' \
    sentence-transformers \
    qdrant-client \
    langchain-text-splitters \
    rank_bm25 \
    pymupdf \
    pythainlp \
    scikit-learn \
    rich

# English cache English docling-models English ONNX file mismatch
import shutil, os
cache_path = os.path.expanduser('~/.cache/huggingface/hub/models--ds4sd--docling-models')
if os.path.exists(cache_path):
    shutil.rmtree(cache_path)
    print('🗑️ English docling-models cache English')

print('✅ English!')
print('⚠️ English Restart runtime: Runtime → Restart session → English Run English cell English')

  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 1.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 397.4/397.4 kB 25.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 87.4/87.4 kB 6.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 390.4/390.4 kB 26.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.9/24.9 MB 71.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 19.3/19.3 MB 63.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 245.6/245.6 kB 18.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.3/8.3 MB 85.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 34.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.7/42.7 kB 2.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.7/62.7 kB 4.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.0/3.0 MB 81.1 MB/s eta 0:00:00
   ━━━━━━

In [17]:
%%time
# Import libraries English
import hashlib
import os
import json
import numpy as np
from pathlib import Path
from IPython.display import display, Markdown

print('✅ Import English!')

✅ Import English!
CPU times: user 0 ns, sys: 995 µs, total: 995 µs
Wall time: 1.89 ms


### 📂 English

English workshop

In [18]:
%%time
# English
os.makedirs('data', exist_ok=True)
os.makedirs('output', exist_ok=True)

# English: English Case Study English AI English
sample_texts = {

    'case_kmitl.txt': '''English: AI English (KMITL)

English KMITL English AI English Smart Campus
English IoT sensor English Machine Learning English
English 25% English

English English English NLP English
English
English Sentiment Analysis English Topic Modeling English
English

English KMITL English RAG English AI Tutor
English 200 English
English 24 English
English lecture notes, slides, English
''',

    'case_healthcare.txt': '''English: AI English

English AI English (Medical Imaging)
English English X-ray English Deep Learning
English AI English 95% English 92%

English NLP English (EMR)
English
English 15 English 2 English

English: English
English Transfer Learning English
English Fine-tune English''',

    'case_banking.txt': '''English: AI English

English Chatbot English KBTG
English Large Language Model (LLM) English RAG
English

English RAG English
English English English FAQ
English LLM English

English: English Call Center English 40%
English 25%
English 24 English English

English: Vector Database English Embedding
Hybrid Search English Dense + Sparse English''',

    'case_banking_duplicate.txt': '''English: AI English

English Chatbot English KBTG
English Large Language Model (LLM) English RAG
English

English RAG English
English English English FAQ
English LLM English

English: English Call Center English 40%
English 25%
English 24 English English

English: Vector Database English Embedding
Hybrid Search English Dense + Sparse English''',

    'case_education.txt': '''English: AI English

English AI English
English English Intelligent Tutoring System English

English RAG English-English
English English Slides English
English Embed English Vector English Vector Database

English English
English LLM English

English:
1. English: English PDF English Markdown
2. English: English Chunking English
3. English Embedding: English Chunk English Vector English
4. English: Upsert Vector English Qdrant English Vector DB English
5. English: English Hybrid Search English Chunk English
6. English: English Context English LLM English

English: English 24 English
English English 60%''',

    'case_agriculture.txt': '''English: AI English

Smart Farming English AI English
English IoT Sensor English

English Computer Vision English
English Convolutional Neural Network (CNN) English Train English
English 50,000 English English 8 English

English NLP English
English English English Social Media
English

English AI English:
- English
- English
- English Data Engineering English'''
}

for fname, content in sample_texts.items():
    with open(f'data/{fname}', 'w', encoding='utf-8') as f:
        f.write(content)

print(f'✅ English {len(sample_texts)} English English data/')
print()
for fname in sorted(sample_texts.keys()):
    print(f'  📄 {fname}')

✅ English 5 English English data/

  📄 case_agriculture.txt
  📄 case_banking.txt
  📄 case_banking_duplicate.txt
  📄 case_education.txt
  📄 case_healthcare.txt
CPU times: user 12 µs, sys: 999 µs, total: 1.01 ms
Wall time: 2.05 ms


---
## 🔍 Section 1.1: English Duplicate English MD5

### MD5 Hash English?

**MD5 (Message Digest Algorithm 5)** English English hash English 128-bit (32 English hex)

- English → English hash English **English**
- English → English hash English **English**

### English Duplicate?

English RAG English:
- ❌ English storage English embedding
- ❌ English
- ❌ LLM English

In [19]:
%%time
def compute_md5(filepath):
    """English MD5 hash English"""
    md5_hash = hashlib.md5()
    with open(filepath, 'rb') as f:
        for chunk in iter(lambda: f.read(4096), b''):
            md5_hash.update(chunk)
    return md5_hash.hexdigest()

# English: English MD5 English
print('📊 MD5 Hash English:')
print('=' * 70)
file_hashes = {}
for fname in sorted(os.listdir('data')):
    fpath = f'data/{fname}'
    if os.path.isfile(fpath):
        h = compute_md5(fpath)
        file_hashes[fname] = h
        print(f'  {fname:<25} → {h}')

print('\n🔍 English: case_banking.txt English case_banking_duplicate.txt English hash English!')

📊 MD5 Hash English:
  case_agriculture.txt      → 1d03b92b2dbb2d89eeb0c053207fe283
  case_banking.txt          → 66160d60bd9880b6b155915e9f0b7c1d
  case_banking_duplicate.txt → 66160d60bd9880b6b155915e9f0b7c1d
  case_education.txt        → 8e47b92d1afdb2f61800890133ffe7f7
  case_healthcare.txt       → 2c56ffd8024fe12efc351c587515339d

🔍 English: case_banking.txt English case_banking_duplicate.txt English hash English!
CPU times: user 0 ns, sys: 942 µs, total: 942 µs
Wall time: 1.76 ms


In [20]:
%%time
def find_duplicates(directory):
    """English MD5"""
    hash_map = {}  # md5 → [list of filepaths]

    for fname in os.listdir(directory):
        fpath = os.path.join(directory, fname)
        if os.path.isfile(fpath):
            h = compute_md5(fpath)
            hash_map.setdefault(h, []).append(fname)

    # English 1 English
    duplicates = {h: files for h, files in hash_map.items() if len(files) > 1}
    return duplicates, hash_map

duplicates, hash_map = find_duplicates('data')

if duplicates:
    print('⚠️ English!')
    for h, files in duplicates.items():
        print(f'\n  Hash: {h}')
        print(f'  English: {", ".join(files)}')
        print(f'  → English: {files[0]}, English: {", ".join(files[1:])}')
else:
    print('✅ English')

⚠️ English!

  Hash: 66160d60bd9880b6b155915e9f0b7c1d
  English: case_banking_duplicate.txt, case_banking.txt
  → English: case_banking_duplicate.txt, English: case_banking.txt
CPU times: user 1 ms, sys: 0 ns, total: 1 ms
Wall time: 7.06 ms


In [21]:
%%time
# English (English)
removed = []
for h, files in duplicates.items():
    for f in files[1:]:  # English
        os.remove(f'data/{f}')
        removed.append(f)

print(f'🗑️ English {len(removed)} English: {", ".join(removed)}')
print(f'📁 English: {sorted(os.listdir("data"))}')

🗑️ English 1 English: case_banking.txt
📁 English: ['case_agriculture.txt', 'case_banking_duplicate.txt', 'case_education.txt', 'case_healthcare.txt']
CPU times: user 0 ns, sys: 900 µs, total: 900 µs
Wall time: 2.41 ms


### 💡 English
- `case_banking.txt` English `case_banking_duplicate.txt` English **hash English** → English 100%
- English English MD5 English **English** English
- English duplicate → English storage + English

> 🎯 English English — MD5 English

### 🧪 English 1.1

English 2-3 English (English) English `find_duplicates()` English

In [ ]:
%%time
# 🧪 English: English duplicate detection

# 💡 Hint:
#   1. English open('data/my_file.txt', 'w')
#   2. English
#   3. English find_duplicates('data') English

# with open('data/test1.txt', 'w') as f:
#     f.write('English')
# with open('data/test2.txt', 'w') as f:
#     f.write('English')  # English!
# find_duplicates('data')

---
## ✂️ Section 1.2: Chunking English

### English Chunk?

- LLM English **context window English** — English 100 English
- Embedding model English **English** (English 512 tokens)
- Chunk English **retrieval English**

### English Chunk English

| English | English | English | English |
|------|---------|-------|--------|
| Fixed-size | English | English, English | English |
| Recursive | English separator English | English fixed | English separators |
| Sentence-based | English | English | English chunk English |
| Semantic | English embedding English | English | English, English |

In [22]:
%%time
# English
all_docs = {}  # English
all_text = ''

for fname in sorted(os.listdir('data')):
    fpath = f'data/{fname}'
    if os.path.isfile(fpath) and fname.endswith('.txt'):
        with open(fpath, 'r', encoding='utf-8') as f:
            content = f.read()
            all_docs[fname] = content
            all_text += '\n\n' + content

print(f'📄 English: {len(all_docs)} English')
print(f'📄 English: {len(all_text)} English')
for fname, content in all_docs.items():
    print(f'  📄 {fname}: {len(content)} English')

📄 English: 4 English
📄 English: 2628 English
  📄 case_agriculture.txt: 629 English
  📄 case_banking_duplicate.txt: 567 English
  📄 case_education.txt: 877 English
  📄 case_healthcare.txt: 547 English
CPU times: user 859 µs, sys: 21 µs, total: 880 µs
Wall time: 5.96 ms


### English 1: Fixed-size Chunking

In [23]:
%%time
def fixed_size_chunk(text, chunk_size=200, overlap=50):
    """English"""
    chunks = []
    start = 0
    while start < len(text):
        end = start + chunk_size
        chunks.append(text[start:end])
        start = end - overlap  # English
    return chunks

fixed_chunks = fixed_size_chunk(all_text, chunk_size=200, overlap=50)
print(f'📊 Fixed-size: English {len(fixed_chunks)} chunks')
print(f'\n--- Chunk English 1 ---')
print(fixed_chunks[0])
print(f'\n--- Chunk English 2 ---')
print(fixed_chunks[1])

📊 Fixed-size: English 18 chunks

--- Chunk English 1 ---


English: AI English

Smart Farming English AI English
English IoT Sensor English

English Computer Vision English

--- Chunk English 2 ---
English

English Computer Vision English
English Convolutional Neural Network (CNN) English Train English
English 50,000 English English 8 English

English NLP English
CPU times: user 205 µs, sys: 18 µs, total: 223 µs
Wall time: 231 µs


### English 2: Recursive Character Chunking (LangChain)

English **English separator English** English chunk English → English

```python
separators = ['\n\n', '\n', '。', r'\. ', ' ', '']
#              ①      ②    ③     ④    ⑤   ⑥
```

| English | Separator | English | English |
|:-----:|-----------|----------|----------|
| ① | `\n\n` | **English** (English) | English |
| ② | `\n` | **English** | English |
| ③ | `。` | **English (English/English)** | English CJK |
| ④ | `\. ` | **English + English** | "Hello. World" → English |
| ⑤ | ` ` | **English** | English |
| ⑥ | `''` | **English** (English) | English |

**English:**
```
English → English \n\n English
                  ↓ English chunk English
                  English \n
                  ↓ English
                  English English
                  ↓ ...
                  English (English)
```

> 💡 **English**: English English separator ` ` English
> English `\n\n` English `\n` English

In [24]:
%%time
from langchain_text_splitters import RecursiveCharacterTextSplitter

recursive_splitter = RecursiveCharacterTextSplitter(
    chunk_size=200,
    chunk_overlap=50,
    separators=['\n\n', '\n', '。', r'\. ', ' ', ''],  # English
)

recursive_chunks = recursive_splitter.split_text(all_text)
print(f'📊 Recursive: English {len(recursive_chunks)} chunks')
print(f'\n--- Chunk English 1 ---')
print(recursive_chunks[0])
print(f'\n--- Chunk English 2 ---')
print(recursive_chunks[1])

📊 Recursive: English 17 chunks

--- Chunk English 1 ---
English: AI English

Smart Farming English AI English
English IoT Sensor English

--- Chunk English 2 ---
English Computer Vision English
English Convolutional Neural Network (CNN) English Train English
English 50,000 English English 8 English
CPU times: user 27.3 s, sys: 2.93 s, total: 30.2 s
Wall time: 55.6 s


### English 3: Sentence-based Chunking

In [10]:
%%time
import re

def sentence_chunk(text, max_sentences=3):
    """English/English (English)"""
    # English newline English
    sentences = re.split(r'\n+', text)
    sentences = [s.strip() for s in sentences if s.strip()]

    chunks = []
    for i in range(0, len(sentences), max_sentences):
        chunk = '\n'.join(sentences[i:i+max_sentences])
        if chunk:
            chunks.append(chunk)
    return chunks

sentence_chunks = sentence_chunk(all_text, max_sentences=3)
print(f'📊 Sentence-based: English {len(sentence_chunks)} chunks')

# English 3 chunks English
for idx in range(min(3, len(sentence_chunks))):
    print(f'\n--- Chunk English {idx+1} ({len(sentence_chunks[idx])} English) ---')
    print(sentence_chunks[idx])

📊 Sentence-based: English 18 chunks

--- Chunk English 1 (150 English) ---
English: AI English
Smart Farming English AI English
English IoT Sensor English

--- Chunk English 2 (166 English) ---
English Computer Vision English
English Convolutional Neural Network (CNN) English Train English
English 50,000 English English 8 English

--- Chunk English 3 (143 English) ---
English NLP English
English English English Social Media
English
CPU times: user 761 µs, sys: 0 ns, total: 761 µs
Wall time: 769 µs


### 📊 English

In [25]:
%%time
print('📊 English Chunks:')
print(f'  Fixed-size (200 chars):     {len(fixed_chunks)} chunks')
print(f'  Recursive (200 chars):      {len(recursive_chunks)} chunks')
print(f'  Sentence-based (3 sent):    {len(sentence_chunks)} chunks')

print('\n📏 English Chunk:')
for name, chunks in [('Fixed', fixed_chunks), ('Recursive', recursive_chunks), ('Sentence', sentence_chunks)]:
    sizes = [len(c) for c in chunks]
    print(f'  {name:<12}: avg={np.mean(sizes):.0f}, min={min(sizes)}, max={max(sizes)} English')

📊 English Chunks:
  Fixed-size (200 chars):     18 chunks
  Recursive (200 chars):      17 chunks
  Sentence-based (3 sent):    18 chunks

📏 English Chunk:
  Fixed       : avg=193, min=78, max=200 English
  Recursive   : avg=156, min=98, max=198 English
  Sentence    : avg=144, min=40, max=187 English
CPU times: user 304 µs, sys: 3 µs, total: 307 µs
Wall time: 297 µs


### 💡 English
- **Fixed-size** English → English English
- **Recursive** English/English → English Fixed
- **Sentence-based** English English chunk English
- **overlap** English

> 🎯 English: **Recursive** English English/English

### 🧪 English 1.2

English `chunk_size` English `overlap` English English?

In [ ]:
%%time
# 🧪 English: English chunk_size English overlap

# 💡 Hint:
#   1. chunk_size=100 vs 500 → English chunks English?
#   2. overlap=0 vs 100 → English?

# for size in [100, 200, 500]:
#     chunks = fixed_size_chunk(all_text, chunk_size=size, overlap=50)
#     print(f'chunk_size={size}: {len(chunks)} chunks')

---
## 🤖 Section 1.3: English Markdown English Gemini

### Gemini Multimodal English?

**Multimodal** = English English + English + PDF + video

```
English PDF → Gemini "English" English
    ↓
English layout: header, English, bullet, English
    ↓
English Markdown English
```

### English vs English

| | English | English |
|---|------|--------|
| ✅ | English layout English | English internet |
| ✅ | English | English API credits |
| ✅ | English/English | English cloud |

> ⚠️ **English** (English, English) → English Docling English (Section 1.4)

In [12]:
%%time
# English Gemini API
# 🔑 English API Key English: https://aistudio.google.com/apikey
#
# English Colab Secrets:
#   1. English 🔑 English sidebar English
#   2. English 'Add new secret'
#   3. English: GOOGLE_API_KEY
#   4. English API Key English copy English
#   5. English toggle 'Notebook access'

try:
    from google.colab import userdata
    GOOGLE_API_KEY = userdata.get('GOOGLE_API_KEY')
    print('✅ English API Key English Colab Secrets English!')
except Exception:
    GOOGLE_API_KEY = input('🔑 English Gemini API Key English: ')

from google import genai
client = genai.Client(api_key=GOOGLE_API_KEY)

# English API English
response = client.models.generate_content(
    model='gemini-2.5-flash',
    contents='English English English'
)
print(f'✅ English Gemini API English: {response.text.strip()}')

✅ English API Key English Colab Secrets English!
✅ English Gemini API English: English
CPU times: user 7.83 s, sys: 446 ms, total: 8.28 s
Wall time: 28 s


### English Sample PDF English

English PDF English

In [26]:
%%time
import fitz  # PyMuPDF
import subprocess

# English Thai font English Colab
subprocess.run(['apt-get', 'install', '-y', '-qq', 'fonts-thai-tlwg'],
               capture_output=True)
print('✅ English Thai font English')

# English path English font English
import glob
thai_fonts = glob.glob('/usr/share/fonts/truetype/tlwg/*.ttf')
THAI_FONT = thai_fonts[0] if thai_fonts else None
print(f'📝 English font: {THAI_FONT}')

def create_sample_pdf(filepath, text, title='English'):
    """English PDF English"""
    doc = fitz.open()
    page = doc.new_page()

    # English font English
    thai_font = fitz.Font(fontfile=THAI_FONT)

    # English
    tw = fitz.TextWriter(page.rect)
    y = 50
    tw.append((50, y), title, font=thai_font, fontsize=18)
    y += 40

    # English
    for line in text.split('\n'):
        if y > 750:  # English
            tw.write_text(page)
            page = doc.new_page()
            tw = fitz.TextWriter(page.rect)
            y = 50
        tw.append((50, y), line, font=thai_font, fontsize=11)
        y += 18

    tw.write_text(page)
    doc.save(filepath)
    doc.close()

# English PDF
with open('data/case_healthcare.txt', 'r', encoding='utf-8') as f:
    text = f.read()

create_sample_pdf('data/sample.pdf', text, 'English: AI English')
print('✅ English sample.pdf English (English!)')

✅ English Thai font English
📝 English font: /usr/share/fonts/truetype/tlwg/Kinnari-BoldItalic.ttf
✅ English sample.pdf English (English!)
CPU times: user 127 ms, sys: 17.1 ms, total: 144 ms
Wall time: 13.1 s


In [27]:
%%time
# English Gemini English PDF English Markdown
try:
    # English Gemini English PDF English Markdown
    import pathlib

    def pdf_to_markdown_gemini(pdf_path, client):
        """English Gemini English PDF English Markdown"""
        # English PDF
        pdf_bytes = pathlib.Path(pdf_path).read_bytes()

        prompt = '''English PDF English Markdown format
    English:
    - English (English, English, English)
    - English heading (#, ##, ###) English
    - English English Markdown table
    - English English bullet points
    - English English'''

        response = client.models.generate_content(
            model='gemini-2.5-flash',
            contents=[
                genai.types.Part.from_bytes(data=pdf_bytes, mime_type='application/pdf'),
                prompt
            ]
        )
        return response.text

    # English
    markdown_output = pdf_to_markdown_gemini('data/sample.pdf', client)
    print('📝 English Markdown English Gemini:')
    print('=' * 60)
    display(Markdown(markdown_output))
except Exception as e:
    print(f'❌ Error: {e}')
    print('💡 English: 1) API Key English  2) Internet English  3) English PDF English')


📝 English Markdown English Gemini:


# English: AI English

## English: AI English

English AI English (Medical Imaging) English English X-ray English Deep Learning English AI English 95% English 92%

English NLP English (EMR) English English 15 English 2 English

English: English English Transfer Learning English English Fine-tune English

CPU times: user 15.2 ms, sys: 2.01 ms, total: 17.2 ms
Wall time: 4.31 s


In [28]:
%%time
# English
with open('output/gemini_output.md', 'w', encoding='utf-8') as f:
    f.write(markdown_output)
print('✅ English output/gemini_output.md')

✅ English output/gemini_output.md
CPU times: user 1.14 ms, sys: 38 µs, total: 1.18 ms
Wall time: 1.27 ms


### 💡 English
- Gemini **English layout** English → English header, English, bullet English
- English **Multimodal** → English PDF English "English" English
- English: English internet + English API

> ⚠️ English (English, English) — English Docling English (Section 1.4)

### 🧪 English 1.3

English upload PDF English (English slides English) English Gemini English Markdown

In [ ]:
%%time
# 🧪 English: English PDF English
# TODO: Upload PDF English English pdf_to_markdown_gemini()



---
## 📄 Section 1.4: English Markdown English Docling

### Docling English?

**[Docling](https://github.com/DS4SD/docling)** English open-source library English IBM
English (PDF, DOCX, PPTX, HTML) English Markdown English JSON

| | Gemini | Docling |
|---|--------|--------|
| **English** | Cloud API (LLM) | Local library |
| **English** | English quota English | English English |
| **English** | English network | English CPU/GPU |
| **English** | English | English offline English |
| **English** | English internet | English |

In [29]:
%%time
from docling.document_converter import DocumentConverter

# English PDF English Docling
converter = DocumentConverter()
result = converter.convert('data/sample.pdf')

docling_markdown = result.document.export_to_markdown()

print('📝 English Markdown English Docling:')
print('=' * 60)
display(Markdown(docling_markdown))

[INFO] 2026-03-06 03:55:08,055 [RapidOCR] base.py:22: Using engine_name: torch
[INFO] 2026-03-06 03:55:08,061 [RapidOCR] device_config.py:57: Using CPU device
[INFO] 2026-03-06 03:55:08,066 [RapidOCR] download_file.py:68: Initiating download: https://www.modelscope.cn/models/RapidAI/RapidOCR/resolve/v3.7.0/torch/PP-OCRv4/det/ch_PP-OCRv4_det_infer.pth
[INFO] 2026-03-06 03:55:08,951 [RapidOCR] download_file.py:82: Download size: 13.83MB
[INFO] 2026-03-06 03:55:09,114 [RapidOCR] download_file.py:95: Successfully saved to: /usr/local/lib/python3.12/dist-packages/rapidocr/models/ch_PP-OCRv4_det_infer.pth
[INFO] 2026-03-06 03:55:09,117 [RapidOCR] main.py:50: Using /usr/local/lib/python3.12/dist-packages/rapidocr/models/ch_PP-OCRv4_det_infer.pth
[INFO] 2026-03-06 03:55:09,382 [RapidOCR] base.py:22: Using engine_name: torch
[INFO] 2026-03-06 03:55:09,383 [RapidOCR] device_config.py:57: Using CPU device
[INFO] 2026-03-06 03:55:09,385 [RapidOCR] download_file.py:68: Initiating download: https://

📝 English Markdown English Docling:


## English: AI English

English: AI English

English AI English (Medical Imaging) English English X-ray English Deep Learning English AI English 95% English 92%

English NLP English (EMR) English English 15 English 2 English

English: English English Transfer Learning English English Fine-tune English

CPU times: user 13 s, sys: 6.69 s, total: 19.7 s
Wall time: 35.3 s


In [30]:
%%time
# English
with open('output/docling_output.md', 'w', encoding='utf-8') as f:
    f.write(docling_markdown)

print('📊 English Gemini vs Docling:')
print(f'  Gemini output:  {len(markdown_output)} English')
print(f'  Docling output: {len(docling_markdown)} English')
print('\n💡 Tip: English English')
print('  - English (English, English) → Gemini')
print('  - English, English offline → Docling')

📊 English Gemini vs Docling:
  Gemini output:  581 English
  Docling output: 579 English

💡 Tip: English English
  - English (English, English) → Gemini
  - English, English offline → Docling
CPU times: user 1.29 ms, sys: 0 ns, total: 1.29 ms
Wall time: 989 µs


### 📊 English Gemini vs Docling

| English | Gemini API | Docling (IBM) |
|-------|-----------|---------------|
| **English** | ⚡ English (English API) | 🐢 English (run local) |
| **English Markdown** | ⭐⭐⭐ English | ⭐⭐ English |
| **English** | ✅ English | ⚠️ English |
| **English** | 💰 English (English quota) | 🆓 English |
| **English Internet** | ✅ English | ❌ English |
| **English/English** | ✅ English | ✅ English |
| **English** | ⚠️ English cloud | ✅ English |
| **English** | Prototype, English | Production, English |

### 🧪 English 1.4

English PDF English Gemini English Docling English

In [ ]:
%%time
# 🧪 English: English Gemini vs Docling English PDF English
# TODO: English



---
## 🔢 Section 1.5: Embedding English (English)

### Embedding English?

**Embedding = English (vector)** English

```
"AI English"      → [0.23, -0.11, 0.45, ...] (1024 English)
"English"   → [0.21, -0.13, 0.42, ...] ← English! (English)
"English"      → [0.89,  0.56, -0.32, ...] ← English (English)
```

### English multilingual?

| Model | English | Vector Size | English |
|-------|:-------:|:-----------:|:--------:|
| `all-MiniLM-L6-v2` | ❌ English | 384 | ⚡ English |
| `multilingual-e5-large` | ✅ English | 1024 | 🐢 English |
| `paraphrase-multilingual` | ✅ English | 768 | ⚡ English |

> 💡 English **multilingual-e5-large** English
> (English fine-tune model English)

In [31]:
%%time
# English Embedding Model (English HuggingFace)
from sentence_transformers import SentenceTransformer
import warnings
warnings.filterwarnings('ignore')

try:
    embed_model = SentenceTransformer('intfloat/multilingual-e5-large')
    test = embed_model.encode('English')
    print(f'✅ English Embedding Model English!')
    print(f'   Model: intfloat/multilingual-e5-large')
    print(f'   Vector size: {len(test)}')
except Exception as e:
    print(f'❌ English Model English: {e}')
    print('💡 English: 1) Internet 2) Disk space (English ~2GB)')

✅ English Embedding Model English!
   Model: intfloat/multilingual-e5-large
   Vector size: 1024
CPU times: user 12.5 s, sys: 16.2 s, total: 28.7 s
Wall time: 38.4 s


In [33]:
%%time
# English embedding English
thai_sentences = [
    'query: English',
    'passage: AI English',
    'passage: Machine Learning English AI English',
    'passage: English English',
    'passage: Vector Database English',
]

# English embeddings
embeddings = embed_model.encode(thai_sentences)

print(f'📊 English embedding English!')
print(f'  English: {len(embeddings)}')
print(f'  English vector: {embeddings.shape[1]} English')
print(f'  English vector (5 English): {embeddings[0][:5]}')

📊 English embedding English!
  English: 5
  English vector: 1024 English
  English vector (5 English): [ 0.02122011 -0.01509202  0.01514856 -0.05054271  0.05155794]
CPU times: user 2.29 s, sys: 36.2 ms, total: 2.32 s
Wall time: 1.52 s


### 📐 English Vectors

English Vector English English **English** English vector English
English 3 English:

---

**1️⃣ Cosine Similarity** — English "English" English 2 vectors
```
                    A · B           English (a₁×b₁ + a₂×b₂ + ...)
Cosine(A, B) = ─────────── = ─────────────────────────────────────────
                |A| × |B|       English A × English B

English:  -1 (English) ← 0 (English) → 1 (English) ✅
```

**2️⃣ Dot Product** — English
```
Dot(A, B) = a₁×b₁ + a₂×b₂ + a₃×b₃ + ...

English:  -∞ ← 0 → +∞ (English = English)
```

**3️⃣ L2 Distance (Euclidean)** — English "English" English 2 English
```
L2(A, B) = √( (a₁-b₁)² + (a₂-b₂)² + (a₃-b₃)² + ... )

English:  0 (English) → +∞ (English)  ⚠️ English = English
```

---

### 🤔 English?

**1️⃣ Cosine Similarity — English RAG / NLP**

English "English" English "English" English vector

```
English: English
  English A: English AI English 10 English   → vector English (English)
  English B: English AI English 1 English → vector English (English)

  Cosine = 0.95 ✅ → English "English" English
  Dot Product = English ❌ → English vector English
```
👉 **English**: English, RAG retrieval, semantic search

---

**2️⃣ Dot Product — English vector English normalize English**

English normalize vector English = 1 English → Dot Product = Cosine (English English)

```
English: English recommendation
  User vector (normalize):    [0.5, 0.7, 0.1]  (English = 1)
  Product A vector (normalize): [0.4, 0.8, 0.2]  (English = 1)

  Dot = 0.5×0.4 + 0.7×0.8 + 0.1×0.2 = 0.78 ← English!
```
👉 **English**: English, vector English normalize English, English production English

---

**3️⃣ L2 Distance — English "English" English**

English English English

```
English: English (Image Clustering)
  English A: [0.2, 0.8, 0.3]
  English B: [0.3, 0.7, 0.4]  → L2 = 0.17 (English ✅)
  English: [0.9, 0.1, 0.8]  → L2 = 1.12 (English ❌)
```
👉 **English**: image similarity, clustering, anomaly detection

---

> 💡 **English**: English RAG → English **Cosine Similarity** English!

In [ ]:
%%time
# ─── English Thai font English matplotlib ───
import subprocess, glob
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm

# English Thai font package (English Colab)
subprocess.run(['apt-get', '-qq', 'install', '-y', 'fonts-thai-tlwg'], capture_output=True)

# register font English English matplotlib
for font_path in glob.glob('/usr/share/fonts/truetype/tlwg/*.ttf'):
    fm.fontManager.addfont(font_path)

plt.rcParams['font.family'] = 'Garuda'
print('✅ Thai font (Garuda) English')

In [34]:
%%time
# === English 3 English — English Heatmap ===
import matplotlib.pyplot as plt
from sklearn.metrics.pairwise import cosine_similarity, euclidean_distances

labels = ['Q: AI English', 'P: AI English CS', 'P: ML English AI', 'P: English', 'P: VectorDB']
short_labels = ['Q: AI?', 'P: AI-CS', 'P: ML-AI', 'P: English', 'P: VecDB']

# English 3 English
cos_sim = cosine_similarity(embeddings)
dot_prod = np.dot(embeddings, embeddings.T)
l2_dist = euclidean_distances(embeddings)

# --- Heatmap Visualization ---
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

matrices = [
    (cos_sim, '1️⃣ Cosine Similarity\n(English = English)', 'YlOrRd', '.3f'),
    (dot_prod, '2️⃣ Dot Product\n(English = English)', 'YlOrRd', '.1f'),
    (l2_dist, '3️⃣ L2 Distance\n(English = English)', 'YlOrRd_r', '.3f'),
]

for ax, (matrix, title, cmap, fmt) in zip(axes, matrices):
    im = ax.imshow(matrix, cmap=cmap, aspect='auto')
    ax.set_xticks(range(len(short_labels)))
    ax.set_yticks(range(len(short_labels)))
    ax.set_xticklabels(short_labels, fontsize=10, rotation=45, ha='right')
    ax.set_yticklabels(short_labels, fontsize=10)
    ax.set_title(title, fontsize=12, fontweight='bold', pad=10)

    # English
    for i in range(len(labels)):
        for j in range(len(labels)):
            val = matrix[i][j]
            color = 'white' if val > (matrix.max() + matrix.min()) / 2 else 'black'
            ax.text(j, i, f'{val:{fmt}}', ha='center', va='center', fontsize=9, color=color)

    plt.colorbar(im, ax=ax, shrink=0.8)

plt.suptitle('📐 English 3 English Vector', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

print('\n💡 English:')
print('  - Cosine: English AI English similarity English (~0.9), English (~0.7)')
print('  - L2: English AI English, English English')
print('  - Dot Product English Cosine English vector English normalize English')

1️⃣ Cosine Similarity (English = English, max=1.0)
                   Q: AI English   P: AI English CSP: ML English AI      P: English     P: VectorDB
   Q: AI English        1.000           0.850           0.818           0.722           0.754   
   P: AI English CS        0.850           1.000           0.917  ⭐        0.796           0.827   
P: ML English AI        0.818           0.917  ⭐        1.000           0.806           0.846   
      P: English        0.722           0.796           0.806           1.000           0.818   
     P: VectorDB        0.754           0.827           0.846           0.818           1.000   


2️⃣ Dot Product (English = English, English max)
                   Q: AI English   P: AI English CSP: ML English AI      P: English     P: VectorDB
   Q: AI English             1.0             0.8             0.8             0.7             0.8
   P: AI English CS             0.8             1.0             0.9             0.8             0.8
P: ML English

### 🎨 Visualization: English = English

English 1024 → 2 English PCA English plot English vector English

In [ ]:
%%time
from sklearn.decomposition import PCA

# English 1024 → 2 English PCA
pca = PCA(n_components=2)
coords = pca.fit_transform(embeddings)

# Plot
plt.figure(figsize=(10, 7))

labels = ['Q: AI English', 'P: AI English CS', 'P: ML English AI', 'P: English', 'P: VectorDB']
colors = ['red', 'blue', 'blue', 'gray', 'green']

for j, (x, y) in enumerate(coords):
    plt.scatter(x, y, c=colors[j], s=200, zorder=5)
    plt.annotate(labels[j], (x, y), textcoords='offset points',
                 xytext=(10, 10), fontsize=11,
                 bbox=dict(boxstyle='round,pad=0.3', facecolor='yellow', alpha=0.7))

# English query English passages
for j in range(1, len(coords)):
    plt.plot([coords[0][0], coords[j][0]], [coords[0][1], coords[j][1]],
             'k--', alpha=0.3)

plt.title('📐 Embedding Space (PCA 2D) — English = English', fontsize=14)
plt.xlabel('PCA Component 1')
plt.ylabel('PCA Component 2')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print('💡 English: English AI (English) English')
print('   English (English) English')

### 💡 English
- English **English** → cosine score **English 1.0**
- English **English** → cosine score **English 0**
- English PCA plot: English **English** → English semantic search
- `multilingual-e5-large` English English train English 100 English

> 🎯 **Embedding English RAG** — English embedding English English

### 🧪 English 1.5

English 5 English English similarity English

In [ ]:
%%time
# 🧪 English: English embedding English
# TODO: English



---
## 🔀 Section 1.6: Hybrid Embedding (Dense + Sparse)

### Dense Embedding English?

**Dense Embedding** English vector English **English** (English 0)
English Neural Network English "English" English

```
"English" → [0.12, -0.34, 0.56, 0.78, 0.23, -0.11, ...] (1024 English, English 0)
```

✅ **English**: English "English" — English "English" English "English" English
❌ **English**: English keyword English English English "SKU-12345"

---

### Sparse Embedding English?

**Sparse Embedding** English vector English **English 0**
English English English BM25 English TF-IDF

```
English vocabulary = [English, English, English, English, English, English, ...] (50,000 English)

"English" → [0.8, 0, 0.5, 0, 0.3, 0, 0, 0, 0, ...] (50,000 English, English 0)
                    English  English English English English English
```

✅ **English**: English keyword English English English English, English, English
❌ **English**: English synonym — English "English" English "English" English

---

### 🔀 Hybrid Search — English?

| English | Dense | Sparse | Hybrid |
|-----------|-------|--------|--------|
| "AI English" → English "English" | ✅ English | ❌ English | ✅ English |
| English "SKU-12345" | ❌ English | ✅ English | ✅ English |
| "English" | ✅ English | ✅ English | ✅✅ English |

**English Hybrid:**
```
Hybrid Score = α × Dense Score + (1-α) × Sparse Score

α = 1.0 → English Dense English
α = 0.5 → English (English)
α = 0.0 → English Sparse English
```

### 🇹🇭 English Tokenizer English?

English **English** English:

```
English:  "I love machine learning"
          → English split English → ["I", "love", "machine", "learning"]  ✅ English!

Thai:     "English"
          → split English → ["English"]  ❌ English!
          → English pythainlp  → ["English", "English", "English", "English", "English", "English"]  ✅
```

**BM25 English token (English) English** → English English!

English `pythainlp.word_tokenize()` English dictionary + ML English

In [36]:
%%time
from rank_bm25 import BM25Okapi
import pythainlp

# English
documents = [
    'English English',
    'Machine Learning English AI English',
    'NLP English',
    'Vector Database English',
    'RAG English LLM English',
    'Qdrant English open-source vector database',
]

# English PyThaiNLP
tokenized_docs = [pythainlp.word_tokenize(doc) for doc in documents]
print('📝 English:')
for i, (doc, tokens) in enumerate(zip(documents[:2], tokenized_docs[:2])):
    print(f'  [{i}] {doc}')
    print(f'      → {tokens}')

# English BM25 index (Sparse)
bm25 = BM25Okapi(tokenized_docs)
print(f'\n✅ English BM25 index English ({len(documents)} documents)')

📝 English:
  [0] English English
      → ['English', ' ', 'English', 'English', 'English', 'English', 'English', 'English']
  [1] Machine Learning English AI English
      → ['Machine', ' ', 'Learning', ' ', 'English', 'English', 'English', ' ', 'AI', ' ', 'English', 'English', 'English', 'English']

✅ English BM25 index English (6 documents)
CPU times: user 2.82 s, sys: 235 ms, total: 3.05 s
Wall time: 3.15 s


In [37]:
%%time
# English Hybrid Search
def hybrid_search(query, documents, embed_model, bm25, tokenized_docs, alpha=0.5, top_k=3):
    """English Hybrid: English Dense + Sparse"""
    # Dense search (Semantic)
    q_emb = embed_model.encode([f'query: {query}'])
    doc_embs = embed_model.encode([f'passage: {d}' for d in documents])
    dense_scores = cosine_similarity(q_emb, doc_embs)[0]

    # Sparse search (BM25)
    q_tokens = pythainlp.word_tokenize(query)
    sparse_scores = bm25.get_scores(q_tokens)

    # Normalize scores to [0, 1]
    if max(dense_scores) > 0:
        dense_norm = dense_scores / max(dense_scores)
    else:
        dense_norm = dense_scores

    if max(sparse_scores) > 0:
        sparse_norm = sparse_scores / max(sparse_scores)
    else:
        sparse_norm = sparse_scores

    # Hybrid score
    hybrid_scores = alpha * dense_norm + (1 - alpha) * sparse_norm

    # English
    ranked = sorted(enumerate(zip(documents, dense_norm, sparse_norm, hybrid_scores)),
                    key=lambda x: x[1][3], reverse=True)
    return ranked[:top_k]

# English
query = 'AI English'
results = hybrid_search(query, documents, embed_model, bm25, tokenized_docs, alpha=0.5)

print(f'🔍 Query: "{query}"')
print(f'{"":>4}{"Document":<50} {"Dense":>8} {"Sparse":>8} {"Hybrid":>8}')
print('=' * 82)
for idx, (doc, dense, sparse, hybrid) in results:
    print(f'  [{idx}] {doc:<48} {dense:>8.3f} {sparse:>8.3f} {hybrid:>8.3f}')

🔍 Query: "AI English"
    Document                                              Dense   Sparse   Hybrid
  [1] Machine Learning English AI English    1.000    1.000    1.000
  [5] Qdrant English open-source vector database             0.919    0.138    0.528
  [3] Vector Database English          0.922    0.110    0.516
CPU times: user 2.79 s, sys: 23.6 ms, total: 2.81 s
Wall time: 2.49 s


In [38]:
%%time
# English alpha English
print('📊 English alpha (Dense weight) English:')
print('=' * 70)
query = 'vector database English RAG'
print(f'Query: "{query}"\n')

for alpha in [0.0, 0.3, 0.5, 0.7, 1.0]:
    results = hybrid_search(query, documents, embed_model, bm25, tokenized_docs, alpha=alpha, top_k=2)
    top_doc = results[0][1][0]
    top_score = results[0][1][3]
    label = 'Sparse only' if alpha == 0 else 'Dense only' if alpha == 1 else f'Hybrid'
    print(f'  α={alpha:.1f} ({label:<12}): [{results[0][0]}] {top_doc[:45]}... ({top_score:.3f})')

📊 English alpha (Dense weight) English:
Query: "vector database English RAG"

  α=0.0 (Sparse only ): [5] Qdrant English open-source vector database... (1.000)
  α=0.3 (Hybrid      ): [5] Qdrant English open-source vector database... (0.994)
  α=0.5 (Hybrid      ): [5] Qdrant English open-source vector database... (0.991)
  α=0.7 (Hybrid      ): [5] Qdrant English open-source vector database... (0.987)
  α=1.0 (Dense only  ): [3] Vector Database English... (1.000)
CPU times: user 13.8 s, sys: 228 ms, total: 14 s
Wall time: 14.4 s


### 💡 English
- **Dense** (embedding) English **English** → "English" = "English"
- **Sparse** (BM25) English **English** → "Python" English
- **alpha=0.7** (Dense 70% + Sparse 30%) English
- English tokenizer English (PyThaiNLP) English

> 🎯 **Hybrid = English + English** → English

### 🧪 English 1.6

English query English English English alpha English

In [ ]:
%%time
# 🧪 English: English Hybrid search English query English

# 💡 Hint:
#   1. English alpha English: 0.0 (Sparse only), 0.5 (50/50), 1.0 (Dense only)
#   2. English query English keyword English vs English

# hybrid_search('Deep Learning', all_chunks, embed_model, bm25,
#               tokenized_docs, alpha=0.5, top_k=3)

---
## 🗄️ Section 1.7: English Qdrant VectorDB

### Vector Database English?

**Vector Database** = English vector English

```
English → Embedding → Vector [0.12, -0.45, 0.78, ...] → English VectorDB
                                                           ↕
Query   → Embedding → Vector → English vector English (ANN)
```

### English Qdrant?

| VectorDB | English | Cloud | In-Memory | Open Source |
|----------|------|:-----:|:---------:|:-----------:|
| **Qdrant** | Rust | ✅ | ✅ | ✅ |
| ChromaDB | Python | ❌ | ✅ | ✅ |
| Pinecone | — | ✅ | ❌ | ❌ |
| Weaviate | Go | ✅ | ✅ | ✅ |

> 💡 English **Qdrant** English (Rust), English in-memory English (English Colab), English filter English payload English

In [44]:
%%time
from qdrant_client import QdrantClient, models

try:
    qdrant = QdrantClient(':memory:')
    print('✅ English Qdrant English! (in-memory mode)')
    print('💡 in-memory = English RAM English Colab English')
except Exception as e:
    print(f'❌ English Qdrant: {e}')

✅ English Qdrant English! (in-memory mode)
💡 in-memory = English RAM English Colab English
CPU times: user 10.1 ms, sys: 0 ns, total: 10.1 ms
Wall time: 12.1 ms


In [47]:
COLLECTION_NAME = 'thai_documents'
VECTOR_SIZE = 1024

In [49]:
%%time
COLLECTION_NAME = 'thai_documents'
VECTOR_SIZE = 1024

# English Collection English English
if qdrant.collection_exists(collection_name=COLLECTION_NAME):
    qdrant.delete_collection(collection_name=COLLECTION_NAME)
    print(f'🗑️ English Collection "{COLLECTION_NAME}" English')

# English Collection
qdrant.create_collection(
    collection_name=COLLECTION_NAME,
    vectors_config=models.VectorParams(
        size=VECTOR_SIZE,
        distance=models.Distance.COSINE,  # English Cosine similarity
    ),
)

# English
info = qdrant.get_collection(COLLECTION_NAME)
print(f'✅ English Collection "{COLLECTION_NAME}" English!')
print(f'  Status: {info.status}')
print(f'  Vector size: {info.config.params.vectors.size}')
print(f'  Distance: {info.config.params.vectors.distance}')
print(f'  Points: {info.points_count}')

🗑️ English Collection "thai_documents" English
✅ English Collection "thai_documents" English!
  Status: green
  Vector size: 1024
  Distance: Cosine
  Points: 0
CPU times: user 999 µs, sys: 0 ns, total: 999 µs
Wall time: 1.01 ms


### 💡 English
- `:memory:` = English → English English Colab English
- `Cosine` = English "English" English vector → English NLP
- Production English → English Qdrant Cloud English Docker + persistent storage

> 🎯 VectorDB English **English vectors English milliseconds** English ANN algorithm

### 🧪 English 1.7

English collection English `my_collection` English distance metric English `EUCLID`

In [ ]:
# 🧪 English: English collection English
# TODO: English



---
## 📥 Section 1.8: English Index (Upsert English)

### Indexing English?

**Indexing** = English chunk + embedding + metadata English VectorDB

```
English chunk:
  1. Embed: English chunk → vector
  2. English Point: id + vector + payload (text, source)
  3. Upsert: English Qdrant (English id English update)
```

| English | English |
|--------|----------|
| **Point** | English Qdrant (id + vector + payload) |
| **Payload** | metadata English vector (text, source, date, ...) |
| **Upsert** | Insert English / Update English |

In [50]:
%%time
# English indexing
# English → chunk English recursive splitter

all_chunks = []
for fname in sorted(os.listdir('data')):
    if not fname.endswith('.txt'):
        continue
    with open(f'data/{fname}', 'r', encoding='utf-8') as f:
        text = f.read()

    chunks = recursive_splitter.split_text(text)
    for i, chunk in enumerate(chunks):
        all_chunks.append({
            'text': chunk,
            'source': fname,
            'chunk_index': i,
        })

print(f'📊 English:')
print(f'  English chunks English: {len(all_chunks)}')
for fname in sorted(os.listdir('data')):
    if fname.endswith('.txt'):
        count = sum(1 for c in all_chunks if c['source'] == fname)
        print(f'  {fname}: {count} chunks')

📊 English:
  English chunks English: 18
  case_agriculture.txt: 4 chunks
  case_banking_duplicate.txt: 4 chunks
  case_education.txt: 6 chunks
  case_healthcare.txt: 4 chunks
CPU times: user 0 ns, sys: 1.51 ms, total: 1.51 ms
Wall time: 1.39 ms


In [42]:
%%time
# English embeddings English chunk
print('⏳ English embeddings...')
chunk_texts = [f"passage: {c['text']}" for c in all_chunks]
chunk_embeddings = embed_model.encode(chunk_texts, show_progress_bar=True)
print(f'✅ English embeddings English! ({len(chunk_embeddings)} vectors × {chunk_embeddings.shape[1]} English)')

⏳ English embeddings...


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

✅ English embeddings English! (18 vectors × 1024 English)
CPU times: user 18.7 s, sys: 266 ms, total: 18.9 s
Wall time: 11.5 s


In [51]:
%%time
# Upsert English Qdrant
points = []
for i, (chunk, embedding) in enumerate(zip(all_chunks, chunk_embeddings)):
    points.append(
        models.PointStruct(
            id=i,
            vector=embedding.tolist(),
            payload={
                'text': chunk['text'],
                'source': chunk['source'],
                'chunk_index': chunk['chunk_index'],
            }
        )
    )

# Upsert English
qdrant.upsert(
    collection_name=COLLECTION_NAME,
    points=points,
)

# English
info = qdrant.get_collection(COLLECTION_NAME)
print(f'✅ Upsert English! English points English collection: {info.points_count}')

✅ Upsert English! English points English collection: 18
CPU times: user 19.8 ms, sys: 814 µs, total: 20.6 ms
Wall time: 21.1 ms


### 💡 English
- **payload** = metadata English vector (English text, source)
- English filter English English English `source='healthcare'`
- **upsert** = insert + update → English id English update English

> 🎯 English payload English → English filter English

### 🧪 English 1.8

English collection (English English)

In [ ]:
# 🧪 English: English
# TODO: English



---
## 🔎 Section 1.9: Retrieve English Qdrant

### Retrieval English?

**Retrieval** = English chunk English query English VectorDB

```
Query: "AI English"
  ↓ embed
Vector: [0.23, -0.11, ...]
  ↓ English Qdrant (Cosine Similarity)
Top-3 Results:
  1. [score=0.92] "English AI English X-ray..."
  2. [score=0.85] "NLP English..."
  3. [score=0.71] "English Deep Learning English..."
```

### English

| Parameter | English | English |
|-----------|--------|----------|
| `top_k` | English chunk English | 3-5 (English/English) |
| `score_threshold` | English | 0.5+ |
| `filter` | English metadata | source, date, category |

### Dense Search (Semantic Search)

### 🎯 top_k English?

**`top_k`** English English score English

```
English: English 100 chunks English database

top_k=3  → English 3 chunks English ✅ English, English
top_k=5  → English 5 chunks English    English
top_k=10 → English 10 chunks English   ⚠️ English noise English
```

**English top_k English:**

| top_k | English | English | English |
|-------|---------|-------|--------|
| 1-3 | English | English, English | English |
| 3-5 | English (English) | English | — |
| 5-10 | English / English | English | English, context English |

> 💡 **English**: top_k=3~5 English RAG
> English LLM English English

In [52]:
%%time
def search_qdrant(query, top_k=3):
    """English Qdrant English Dense embedding"""
    # English query embedding
    q_embedding = embed_model.encode([f'query: {query}'])[0]

    # English
    results = qdrant.query_points(
        collection_name=COLLECTION_NAME,
        query=q_embedding.tolist(),
        limit=top_k,
        with_payload=True,
    )
    return results

# English
query = 'English'
results = search_qdrant(query)

print(f'🔍 Query: "{query}"')
print(f'📊 English Top-3:')
print('=' * 70)
for i, point in enumerate(results.points):
    print(f'\n🏆 English {i+1} (Score: {point.score:.4f})')
    print(f'   English: {point.payload["source"]} (chunk {point.payload["chunk_index"]})')
    print(f'   English: {point.payload["text"][:150]}...')

🔍 Query: "English"
📊 English Top-3:

🏆 English 1 (Score: 0.7913)
   English: case_agriculture.txt (chunk 0)
   English: English: AI English

Smart Farming English AI English
English IoT Sensor English...

🏆 English 2 (Score: 0.7865)
   English: case_healthcare.txt (chunk 0)
   English: English: AI English...

🏆 English 3 (Score: 0.7815)
   English: case_healthcare.txt (chunk 1)
   English: English AI English (Medical Imaging)
English English X-ray English Deep Learning
English...
CPU times: user 543 ms, sys: 14.6 ms, total: 558 ms
Wall time: 369 ms


### English

In [53]:
%%time
# English
test_queries = [
    'Vector Database English',
    'RAG English',
    'Deep Learning English Machine Learning English',
]

for query in test_queries:
    results = search_qdrant(query, top_k=2)
    print(f'\n🔍 Query: "{query}"')
    for i, point in enumerate(results.points):
        print(f'   [{i+1}] (Score: {point.score:.4f}) {point.payload["text"][:80]}...')
    print('-' * 70)


🔍 Query: "Vector Database English"
   [1] (Score: 0.8410) English: Vector Database English Embedding
Hybrid Search English Dense + ...
   [2] (Score: 0.8151) 4. English: Upsert Vector English Qdrant English Vector DB English
5. English: English Hybrid S...
----------------------------------------------------------------------

🔍 Query: "RAG English"
   [1] (Score: 0.8712) English RAG English
English ...
   [2] (Score: 0.8572) English RAG English-English
English English...
----------------------------------------------------------------------

🔍 Query: "Deep Learning English Machine Learning English"
   [1] (Score: 0.8206) English: Vector Database English Embedding
Hybrid Search English Dense + ...
   [2] (Score: 0.8100) 4. English: Upsert Vector English Qdrant English Vector DB English
5. English: English Hybrid S...
----------------------------------------------------------------------
CPU times: user 1.57 s, sys: 33.1 ms, total: 1.6 s
Wall time: 1.03 s


### English Filter English Metadata

In [54]:
%%time
# English
query = 'English'
q_embedding = embed_model.encode([f'query: {query}'])[0]

results_filtered = qdrant.query_points(
    collection_name=COLLECTION_NAME,
    query=q_embedding.tolist(),
    query_filter=models.Filter(
        must=[
            models.FieldCondition(
                key='source',
                match=models.MatchValue(value='case_healthcare.txt'),
            )
        ]
    ),
    limit=3,
    with_payload=True,
)

print(f'🔍 Query: "{query}" (filter: source=case_healthcare.txt)')
for i, point in enumerate(results_filtered.points):
    print(f'   [{i+1}] ({point.score:.4f}) [{point.payload["source"]}] {point.payload["text"][:80]}...')

🔍 Query: "English" (filter: source=case_healthcare.txt)
   [1] (0.7978) [case_healthcare.txt] English: AI English...
   [2] (0.7856) [case_healthcare.txt] English: English
English Transfer Learning...
   [3] (0.7817) [case_healthcare.txt] English AI English (Medical Imaging)...
CPU times: user 486 ms, sys: 10.2 ms, total: 496 ms
Wall time: 312 ms


### 💡 English
- score English **1.0** = English, English **0** = English
- query **English text** English → "AI English" English "English" English
- `top_k` English (3-5) → English English | English (10-20) → English English noise
- **Filter** English → English + English

> 🎯 **Retrieval English RAG** — English = English

### 🧪 English 1.9

1. English
2. English filter English
3. English `top_k` English

In [ ]:
%%time
# 🧪 English: English Qdrant

# 💡 Hint:
#   1. English query English vs English
#   2. English top_k → English?
#   3. English filter English source English

# search_qdrant('English', top_k=5)
# search_qdrant('AI', top_k=3)  # English?

---
## 🚀 Bonus: End-to-End RAG Demo

English English retrieved chunks English Gemini English!

```
English → Retrieval (Qdrant) → Top Chunks → LLM (Gemini) → English
```

> 💡 English **Day 2** English Agent English!

In [56]:
%%time
def rag_answer(question, top_k=3):
    """English RAG English: English + English"""
    # 1. Embed query
    query_vector = embed_model.encode(f'query: {question}').tolist()

    # 2. Retrieve from Qdrant
    results = qdrant.query_points(
        collection_name='thai_documents',
        query=query_vector,
        limit=top_k
    ).points

    # 3. English context English chunks English
    context_parts = []
    print(f'🔍 English: {question}')
    print(f'📚 Retrieved {len(results)} chunks:')
    for idx, r in enumerate(results):
        text = r.payload['text']
        context_parts.append(text)
        print(f'  [{idx+1}] score={r.score:.4f} | {text[:60]}...')

    context = '\n\n'.join(context_parts)

    # 4. English Gemini English
    prompt = f'''English English English English
English "English"

English:
{context}

English: {question}
English:'''

    response = client.models.generate_content(
        model='gemini-2.5-flash',
        contents=prompt
    )

    print(f'\n🤖 English RAG:')
    print(f'   {response.text.strip()}')
    print()
    return response.text.strip()

# English
questions = [
    'AI English',
    'RAG English',
    'English AI English',
    'English AI English'
]

for q in questions:
    print('=' * 60)
    rag_answer(q)
    print()

🔍 English: AI English
📚 Retrieved 3 chunks:
  [1] score=0.8733 | English AI English...
  [2] score=0.8680 | English: AI English...
  [3] score=0.8512 | English: AI English

English...

🤖 English RAG:
   AI English English English X-ray English


🔍 English: RAG English
📚 Retrieved 3 chunks:
  [1] score=0.8503 | English: AI English

English...
  [2] score=0.8480 | English RAG English-English
English...
  [3] score=0.8456 | English RAG English
English...

🤖 English RAG:
   RAG English (English English Chatbot KBTG) English English English English English English FAQ English LLM English


🔍 English: English AI English
📚 Retrieved 3 chunks:
  [1] score=0.8596 | English:
1. English: English PDF English...
  [2] score=0.8370 | English: AI English

English...
  [3] score=0.8259 | 4. English: Upsert Vector English Qdrant English Vector DB English
5....

🤖 English RAG:
   English


🔍 English: English AI English
📚 Retrieved 3 chunks:
  [1] score=0.8675 | English: AI English

English...
  [2] scor

---
## 🎯 English: English

### Pipeline English Data Engineering English RAG

| Step | Section | Task |
|:----:|---------|------|
| 📄 | Input | Raw Documents (PDF, TXT, DOCX) |
| 🔍 | 1.1 | English Duplicate (MD5) → English |
| 📝 | 1.3-1.4 | English Markdown (Gemini / Docling) |
| ✂️ | 1.2 | Chunking (Fixed / Recursive / Sentence) |
| 🔢 | 1.5 | Dense Embedding (multilingual-e5-large) |
| 🔀 | 1.6 | Hybrid Embedding (Dense + BM25) |
| 🗄️ | 1.7 | English Qdrant Collection |
| 📥 | 1.8 | Index (Upsert vectors + metadata) |
| 🔎 | 1.9 | Retrieve (Dense / Filtered search) |


### 🔑 Key Takeaways

1. **English** — "Garbage in, garbage out"
2. **Chunking strategy** English RAG
3. **Hybrid search** (Dense + Sparse) English
4. **Metadata** English filter English

### 📅 English: Day 2 — Building Agents: Finding the Needle in the Haystack

English **AI Agents** English RAG pipeline English
English!